In [ ]:
SELECT table_name, column_name
FROM spark_catalog.silver.information_schema.columns
WHERE lower(column_name) LIKE '%service%'
   OR lower(column_name) LIKE '%clienttype%'
   OR lower(column_name) LIKE '%tenancy%'
ORDER BY table_name, column_name;

In [ ]:
SELECT DISTINCT trim(clienttype) AS v
FROM <that_table>
WHERE clienttype IS NOT NULL AND trim(clienttype) <> ''
LIMIT 50;

In [ ]:
# Search columns across all tables in a database (Lakehouse schema)
db = "silver"
patterns = ["service", "clienttype", "tenancy"]

tables = [t.name for t in spark.catalog.listTables(db) if t.tableType.lower() != "view"]
hits = []

for tbl in tables:
    cols = [c.name.lower() for c in spark.table(f"{db}.{tbl}").schema.fields]
    if any(any(p in col for p in patterns) for col in cols):
        for c in cols:
            if any(p in c for p in patterns):
                hits.append((tbl, c))

display(spark.createDataFrame(hits, ["table_name", "column_name"]).orderBy("table_name", "column_name"))

--------------new------------------

In [ ]:
%sql
DROP TABLE IF EXISTS silver.silver_rdm_service_add;

CREATE TABLE silver.silver_rdm_service_add AS
WITH src AS (
    SELECT DISTINCT
        cp.z_src_system_id,
        cp.z_src_system_instance,
        trim(cp.cprod_service)        AS service_src_name_raw,
        trim(lower(cp.cprod_service)) AS service_src_name_norm
    FROM silver.silver_care_product cp
    WHERE cp.cprod_service IS NOT NULL
      AND trim(cp.cprod_service) <> ''
),
rdm AS (
    SELECT DISTINCT
        r.z_src_system_id,
        r.z_src_system_instance,
        trim(lower(r.service_src_name)) AS service_src_name_norm
    FROM silver.silver_rdm_service r
    WHERE r.service_src_name IS NOT NULL
      AND trim(r.service_src_name) <> ''
)
SELECT
    -- candidate source value
    s.service_src_name_raw AS service_src_name,

    -- temp: keep same as name for now
    s.service_src_name_raw AS service_src_id,

    -- business fills later
    'Unknown' AS service_name_conformed,
    'Unknown' AS service_mstr_service_id,
    'Unknown' AS service_bu_id,

    -- lineage
    s.z_src_system_id       AS z_src_system_id,
    s.z_src_system_instance AS z_src_system_instance,

    -- active flag
    1 AS z_order_is_active,

    -- audit
    current_timestamp() AS z_src_created_date_time,
    current_user()      AS z_src_created_by_user,
    current_timestamp() AS z_src_modified_date_time,
    current_user()      AS z_src_modified_by_user
FROM src s
LEFT JOIN rdm r
  ON s.z_src_system_id = r.z_src_system_id
 AND s.z_src_system_instance = r.z_src_system_instance
 AND s.service_src_name_norm = r.service_src_name_norm
WHERE r.service_src_name_norm IS NULL;

In [ ]:
%sql
SELECT count(*) FROM silver.silver_rdm_service_add;

%sql
SELECT * FROM silver.silver_rdm_service_add LIMIT 50;